# Walmart Store Sales Forecasting — CatBoost Experiment

**Tree-Based Model**

CatBoost is a gradient-boosted decision-tree library developed by Yandex. Its key advantage over XGBoost and LightGBM is **native ordered categorical encoding**: instead of requiring manual one-hot or label encoding, CatBoost builds an internal statistic (similar to target encoding with fold-based ordering to prevent leakage) for every categorical feature. This makes it particularly well-suited for this dataset where `Store`, `Dept`, and store `Type` are natural categoricals.

MLflow Experiment structure:
- **CatBoost_Training** ← experiment
  - `CatBoost_Cleaning`
  - `CatBoost_Feature_Selection`
  - `CatBoost_CV`
  - `CatBoost_Tuning`

In [ ]:
import sys
print(sys.executable)

## Setup

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import catboost as cb
from catboost import CatBoostRegressor
import mlflow
import mlflow.catboost
import dagshub

# local src modules
from src.wmae import wmae
from src.cv_split import time_series_cv_splits, last_cutoff_split
from src.features import (
    load_raw_data, merge_all, build_all_features, add_origin_style_features, build_origin_training_frame,
    BASE_FEATURES, LAG_FEATURES_ORIGIN,
    ALL_FEATURES_ORIGIN,
    CATBOOST_CAT_FEATURES, TARGET
)

SEED = 42
DATA_DIR = PROJECT_ROOT
np.random.seed(SEED)
print('CatBoost version:', cb.__version__)
print('Project root:', PROJECT_ROOT)
print('All imports successful')


In [ ]:
# DagsHub / MLflow setup
# Keep credentials outside the notebook: set MLFLOW_TRACKING_USERNAME and
# MLFLOW_TRACKING_PASSWORD in your environment before running this cell.
os.environ.setdefault('MLFLOW_TRACKING_USERNAME', 'sansi23')

dagshub.init(
    repo_owner='sansi23',
    repo_name='Walmart-Recruiting---Store-Sales-Forecasting',
    mlflow=True
)

mlflow.set_experiment('CatBoost_Training')
print(mlflow.get_tracking_uri())

## Data Loading

In [ ]:
train_raw, test_raw, features_raw, stores_raw = load_raw_data(DATA_DIR)
train_raw, test_raw = merge_all(train_raw, test_raw, features_raw, stores_raw)

print(f'Train shape : {train_raw.shape}')
print(f'Test  shape : {test_raw.shape}')
print(f'Date range  : {train_raw["Date"].min()} → {train_raw["Date"].max()}')
TEST_HORIZON_WEEKS = test_raw['Date'].nunique()
print(f'Kaggle test horizon: {TEST_HORIZON_WEEKS} weeks')
train_raw.head()

## Exploratory Data Analysis

In [ ]:
# Weekly sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train_raw[TARGET], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Weekly Sales Distribution')
axes[0].set_xlabel('Weekly Sales')
neg_pct = (train_raw[TARGET] < 0).mean() * 100
axes[0].axvline(0, color='red', linestyle='--', label=f'Negative: {neg_pct:.1f}%')
axes[0].legend()

axes[1].hist(np.log1p(train_raw[TARGET].clip(lower=0)), bins=80, color='coral', edgecolor='white')
axes[1].set_title('log1p(Weekly Sales) Distribution (clipped at 0)')
axes[1].set_xlabel('log1p(Weekly Sales)')
plt.tight_layout()
plt.show()
print(f'Negative sales rows: {(train_raw[TARGET] < 0).sum()}')

In [ ]:
# Sales by store type
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, store_type in zip(axes, ['A', 'B', 'C']):
    subset = train_raw[train_raw['Type'] == store_type]
    weekly = subset.groupby('Date')[TARGET].sum()
    ax.plot(weekly.index, weekly.values, linewidth=1)
    ax.set_title(f'Type {store_type} — Total Weekly Sales')
    ax.set_xlabel('Date')
    for tick in ax.get_xticklabels():
        tick.set_rotation(30)
plt.tight_layout()
plt.show()

In [ ]:
# Holiday vs non-holiday sales
holiday_avg    = train_raw[train_raw['IsHoliday'] == True][TARGET].mean()
nonholiday_avg = train_raw[train_raw['IsHoliday'] == False][TARGET].mean()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Holiday', 'Non-Holiday'], [holiday_avg, nonholiday_avg], color=['tomato', 'steelblue'])
ax.set_title('Average Weekly Sales: Holiday vs Non-Holiday')
ax.set_ylabel('Avg Weekly Sales')
plt.tight_layout()
plt.show()
print(f'Holiday avg: {holiday_avg:,.0f}  |  Non-holiday avg: {nonholiday_avg:,.0f}')

In [ ]:
# Missing values heatmap
missing = train_raw.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]
fig, ax = plt.subplots(figsize=(8, 4))
missing.plot.bar(ax=ax, color='salmon')
ax.set_title('Missing Value Rate per Column')
ax.set_ylabel('Fraction Missing')
ax.axhline(0.5, color='red', linestyle='--', label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()
print(missing)

In [ ]:
# Seasonality: average sales by week-of-year
weekly_avg = train_raw.groupby(train_raw['Date'].dt.isocalendar().week.rename('Week'))[TARGET].mean()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(weekly_avg.index, weekly_avg.values, marker='o', markersize=3)
ax.set_title('Average Weekly Sales by Week-of-Year (seasonality)')
ax.set_xlabel('Week of Year')
ax.set_ylabel('Average Weekly Sales')
plt.tight_layout()
plt.show()

### EDA Takeaways for CatBoost

The charts above suggest several modeling choices for CatBoost:

- Weekly sales are highly right-skewed and include a small number of negative rows, so MAE/WMAE is a better optimization target than RMSE.
- Store type and department patterns differ strongly, so `Store`, `Dept`, and `Type_Enc` should be categorical features rather than simple continuous variables.
- Holiday weeks have different behavior and receive 5x weight in the competition metric, so holiday flags and holiday-proximity features should stay in the model.
- MarkDown columns have structured missingness, which usually means no promotion was active rather than random missing data. Filling them with 0 and adding total/count features is appropriate.
- Week-of-year seasonality is visible, so calendar features and lag/rolling sales history are important for CatBoost.


In [ ]:
# CatBoost-focused EDA: strongest categorical and promotion signals
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Average sales by store type
store_type_sales = train_raw.groupby('Type')[TARGET].mean().sort_values(ascending=False)
store_type_sales.plot.bar(ax=axes[0, 0], color=['steelblue', 'coral', 'seagreen'])
axes[0, 0].set_title('Average Weekly Sales by Store Type')
axes[0, 0].set_xlabel('Store Type')
axes[0, 0].set_ylabel('Average Weekly Sales')
axes[0, 0].tick_params(axis='x', rotation=0)

# 2. Top departments by average weekly sales
top_depts = train_raw.groupby('Dept')[TARGET].mean().sort_values(ascending=False).head(15)
top_depts.sort_values().plot.barh(ax=axes[0, 1], color='slateblue')
axes[0, 1].set_title('Top 15 Departments by Average Weekly Sales')
axes[0, 1].set_xlabel('Average Weekly Sales')
axes[0, 1].set_ylabel('Department')

# 3. Markdown availability over time
md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
markdown_by_date = train_raw.assign(
    HasMarkdown=train_raw[md_cols].fillna(0).gt(0).any(axis=1)
).groupby('Date')['HasMarkdown'].mean()
axes[1, 0].plot(markdown_by_date.index, markdown_by_date.values, color='darkorange')
axes[1, 0].set_title('Share of Rows with Any MarkDown by Date')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Share with Markdown')
for tick in axes[1, 0].get_xticklabels():
    tick.set_rotation(30)

# 4. Total weekly sales with holiday weeks highlighted
weekly_total = train_raw.groupby('Date')[TARGET].sum()
holiday_dates = train_raw.loc[train_raw['IsHoliday'], 'Date'].drop_duplicates().sort_values()
axes[1, 1].plot(weekly_total.index, weekly_total.values, color='steelblue', linewidth=1)
axes[1, 1].scatter(
    holiday_dates,
    weekly_total.reindex(holiday_dates),
    color='tomato', s=35, label='Holiday week', zorder=3
)
axes[1, 1].set_title('Total Weekly Sales with Holiday Weeks Highlighted')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Total Weekly Sales')
axes[1, 1].legend()
for tick in axes[1, 1].get_xticklabels():
    tick.set_rotation(30)

plt.tight_layout()
plt.show()

### Modeling Implication

These extra checks support using CatBoost as a strong tabular baseline: the target varies by categorical groups (`Store`, `Dept`, `Type_Enc`), has promotion-related effects from MarkDown fields, and shows calendar/holiday seasonality. The feature pipeline in `src/features.py` is designed around exactly those signals.


## Data Cleaning

In [ ]:
with mlflow.start_run(run_name='CatBoost_Cleaning'):
    train_clean = train_raw.copy()
    test_clean  = test_raw.copy()

    # 1. Negative weekly sales — these represent returns/corrections.
    #    We keep them as-is for training because CatBoost can learn the pattern,
    #    but we flag how many there are.
    neg_count = (train_clean[TARGET] < 0).sum()
    neg_pct   = neg_count / len(train_clean) * 100

    # 2. MarkDown NA → 0 (no markdown active)
    md_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
    for col in md_cols:
        train_clean[col] = train_clean[col].fillna(0.0).clip(lower=0.0)
        test_clean[col]  = test_clean[col].fillna(0.0).clip(lower=0.0)

    # 3. Economic features: CPI, Unemployment, Temperature, Fuel_Price
    #    Forward-fill within each store (same store = same region)
    eco_cols = ['CPI', 'Unemployment', 'Temperature', 'Fuel_Price']
    for col in eco_cols:
        train_clean[col] = train_clean.groupby('Store')[col].transform(lambda x: x.ffill().bfill())
        test_clean[col]  = test_clean.groupby('Store')[col].transform(lambda x: x.ffill().bfill())

    # 4. Remove duplicate (Store, Dept, Date) rows if any
    dup_count = train_clean.duplicated(subset=['Store','Dept','Date']).sum()
    train_clean = train_clean.drop_duplicates(subset=['Store','Dept','Date'])

    mlflow.log_params({
        'negative_sales_count': int(neg_count),
        'negative_sales_pct':   round(float(neg_pct), 4),
        'duplicate_rows':       int(dup_count),
        'train_rows_after':     len(train_clean),
    })
    remaining_na = train_clean[eco_cols].isnull().sum().to_dict()
    mlflow.log_params({f'na_after_{k}': int(v) for k, v in remaining_na.items()})

    print(f'Negative sales: {neg_count} ({neg_pct:.2f}%)')
    print(f'Duplicates removed: {dup_count}')
    print(f'Train rows after cleaning: {len(train_clean)}')

## Feature Engineering

In [ ]:

ORIGIN_HORIZON_WEEKS = TEST_HORIZON_WEEKS
# Stride 4 gives more historical origins, but was too slow in this environment.
ORIGIN_STRIDE_WEEKS = 13

# Build final-training rows exactly like a real 39-week forecast problem: each
# row is generated from a historical forecast origin, using only sales known at
# that origin. Do not use build_all_features(..., add_lags=True) for final
# Kaggle-style CatBoost training.
train_fe = build_origin_training_frame(
    train_clean,
    horizon_weeks=ORIGIN_HORIZON_WEEKS,
    origin_stride_weeks=ORIGIN_STRIDE_WEEKS,
    min_history_weeks=52,
    require_full_horizon=True,
)

# Real test features use the final training week as the forecast origin.
test_base = build_all_features(test_clean, add_lags=False)
test_fe = add_origin_style_features(test_base, train_clean, origin_date=train_clean['Date'].max())

ORIGIN_FEATURES = [c for c in ALL_FEATURES_ORIGIN if c in train_fe.columns and c in test_fe.columns]
FORBIDDEN_FINAL_FEATURES = {
    'Sales_Lag1', 'Sales_Lag2', 'Sales_Lag4', 'Sales_Lag13', 'Sales_Lag26',
    'Sales_Roll_4w_Mean', 'Sales_Roll_13w_Mean',
    'Sales_Roll_26w_Mean', 'Sales_Roll_52w_Mean',
}
assert not (FORBIDDEN_FINAL_FEATURES & set(ORIGIN_FEATURES)), 'Forbidden per-row lag features entered origin feature set.'

print(f'Origin horizon weeks: {ORIGIN_HORIZON_WEEKS}')
print(f'Origin stride weeks : {ORIGIN_STRIDE_WEEKS}')
print(f'Train after origin feature engineering: {train_fe.shape}')
print(f'Test  after origin feature engineering: {test_fe.shape}')
print('Training forecast origins:', train_fe['Forecast_Origin_Date'].nunique())
print('Origin feature columns:', ORIGIN_FEATURES)
print('Missing origin feature columns in train:', [c for c in ALL_FEATURES_ORIGIN if c not in train_fe.columns])
print('Missing origin feature columns in test :', [c for c in ALL_FEATURES_ORIGIN if c not in test_fe.columns])
train_fe[ORIGIN_FEATURES + [TARGET, 'Forecast_Origin_Date']].head()


## Feature Selection

In [ ]:

with mlflow.start_run(run_name='CatBoost_Feature_Selection'):
    train_model = train_fe.dropna(subset=[TARGET, 'Sales_Lag52_Origin']).reset_index(drop=True)

    # Keep the full origin-safe feature set. Target-based feature selection on the
    # whole training span can make CV slightly optimistic, and CatBoost handles this
    # feature count comfortably.
    model_features = [c for c in ALL_FEATURES_ORIGIN if c in train_model.columns]
    selected_features = model_features.copy()
    assert not (FORBIDDEN_FINAL_FEATURES & set(selected_features)), 'Forbidden per-row lag features selected.'

    X = train_model[model_features]
    y = train_model[TARGET]
    is_holiday = train_model['IsHoliday']
    sample_weight = np.where(is_holiday.astype(bool), 5.0, 1.0)

    print('Training features:', model_features)
    print('Train shape:', X.shape)
    print('Missing values per feature:')
    print(X.isna().sum().sort_values(ascending=False).head(20))

    sample_size = max(1, int(len(X) * 0.2))
    sample_idx = np.random.RandomState(SEED).choice(len(X), size=sample_size, replace=False)
    X_sample = X.iloc[sample_idx]
    y_sample = y.iloc[sample_idx]
    w_sample = sample_weight[sample_idx]

    cat_feature_indices = [X_sample.columns.get_loc(c) for c in CATBOOST_CAT_FEATURES if c in X_sample.columns]

    probe_model = CatBoostRegressor(
        iterations=300, depth=5, learning_rate=0.1,
        loss_function='MAE', random_seed=SEED, verbose=False
    )
    probe_model.fit(X_sample, y_sample, cat_features=cat_feature_indices, sample_weight=w_sample)

    fi = pd.Series(probe_model.get_feature_importance(), index=X_sample.columns)
    fi_sorted = fi.sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 7))
    fi_sorted.head(25).sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_title('CatBoost Feature Importance (top 25)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('catboost_feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()

    num_cols = [c for c in BASE_FEATURES if c in train_model.columns and c not in CATBOOST_CAT_FEATURES]
    corr = train_model[num_cols].corr().abs()
    fig2, ax2 = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, cmap='coolwarm', ax=ax2, vmax=1, vmin=0, center=0.5)
    ax2.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.savefig('catboost_correlation.png', dpi=120, bbox_inches='tight')
    plt.show()

    mlflow.log_params({
        'n_features_before': len(model_features),
        'n_features_selected': len(selected_features),
        'feature_selection_policy': 'all_origin_safe_features',
        'origin_training_rows': len(train_model),
        'origin_training_origins': int(train_model['Forecast_Origin_Date'].nunique()),
    })
    mlflow.log_metric('top_feature_importance', float(fi_sorted.iloc[0]))
    mlflow.log_artifact('catboost_feature_importance.png')
    mlflow.log_artifact('catboost_correlation.png')

    print(f'Using all {len(selected_features)} origin-safe features')
    print('Top 10:', fi_sorted.head(10).index.tolist())


## Cross Validation

In [ ]:

def holiday_weights(df):
    return np.where(df['IsHoliday'].astype(bool), 5.0, 1.0)


with mlflow.start_run(run_name='CatBoost_CV'):
    cutoff = pd.Timestamp('2012-01-27')

    hist = train_clean[train_clean['Date'] <= cutoff].copy()
    val = train_clean[
        (train_clean['Date'] > cutoff) &
        (train_clean['Date'] <= cutoff + pd.Timedelta(weeks=ORIGIN_HORIZON_WEEKS))
    ].copy()

    train_cv = build_origin_training_frame(
        hist,
        horizon_weeks=ORIGIN_HORIZON_WEEKS,
        origin_stride_weeks=ORIGIN_STRIDE_WEEKS,
        min_history_weeks=52,
        require_full_horizon=True,
    )
    train_cv = train_cv.dropna(subset=[TARGET, 'Sales_Lag52_Origin']).reset_index(drop=True)

    val_base = build_all_features(val, add_lags=False)
    val_fe = add_origin_style_features(val_base, hist, origin_date=hist['Date'].max())

    CV_FEATURES = [c for c in ALL_FEATURES_ORIGIN if c in train_cv.columns and c in val_fe.columns]
    assert not (FORBIDDEN_FINAL_FEATURES & set(CV_FEATURES)), 'Forbidden per-row lag features entered CV_FEATURES.'

    X_tr_t = train_cv[CV_FEATURES]
    y_tr_t = train_cv[TARGET]
    w_tr_t = holiday_weights(train_cv)
    X_val_t = val_fe[CV_FEATURES]
    y_val_t = val_fe[TARGET]
    hol_t = val_fe['IsHoliday']
    w_val_t = holiday_weights(val_fe)

    cat_idx_cv = [CV_FEATURES.index(c) for c in CATBOOST_CAT_FEATURES if c in CV_FEATURES]

    print('Cutoff:', cutoff.date())
    print('Train CV history:', hist['Date'].min().date(), '->', hist['Date'].max().date())
    print('Validation:', val['Date'].min().date(), '->', val['Date'].max().date())
    print('Training features:', CV_FEATURES)
    print('Train shape:', X_tr_t.shape)
    print('Validation shape:', X_val_t.shape)
    print('Train missing values per feature:')
    print(X_tr_t.isna().sum().sort_values(ascending=False).head(20))
    print('Validation missing values per feature:')
    print(X_val_t.isna().sum().sort_values(ascending=False).head(20))

    model_cv = CatBoostRegressor(
        iterations=800, depth=6, learning_rate=0.05,
        loss_function='MAE', random_seed=SEED, verbose=False,
        early_stopping_rounds=50,
    )
    model_cv.fit(
        X_tr_t, y_tr_t,
        cat_features=cat_idx_cv,
        sample_weight=w_tr_t,
        eval_set=cb.Pool(X_val_t, y_val_t, cat_features=cat_idx_cv, weight=w_val_t),
    )

    val_preds = model_cv.predict(X_val_t)
    cutoff_wmae = wmae(y_val_t.values, val_preds, hol_t.values)

    hol_mask = hol_t.values.astype(bool)
    mae_hol = np.mean(np.abs(y_val_t.values[hol_mask] - val_preds[hol_mask])) if hol_mask.any() else np.nan
    mae_nhol = np.mean(np.abs(y_val_t.values[~hol_mask] - val_preds[~hol_mask])) if (~hol_mask).any() else np.nan

    cv_df = pd.DataFrame([{
        'cutoff': cutoff,
        'wmae': cutoff_wmae,
        'mae_holiday': mae_hol,
        'mae_nonholiday': mae_nhol,
        'train_rows': len(train_cv),
        'val_rows': len(val_fe),
    }])

    mlflow.log_metric('cutoff_wmae', cutoff_wmae)
    mlflow.log_metric('cutoff_mae_holiday', mae_hol)
    mlflow.log_metric('cutoff_mae_nhol', mae_nhol)
    mlflow.log_param('cv_cutoff', str(cutoff.date()))
    mlflow.log_param('cv_val_weeks', ORIGIN_HORIZON_WEEKS)
    mlflow.log_param('cv_training_feature_contract', 'origin_style')
    mlflow.log_param('cv_origin_stride_weeks', ORIGIN_STRIDE_WEEKS)

    print(f'Cutoff validation WMAE: {cutoff_wmae:,.2f}')
    print(cv_df)


## Hyperparameter Tuning

In [ ]:
with mlflow.start_run(run_name='CatBoost_Tuning'):
    from itertools import product

    # Depth 10 is included because the separate depth experiment showed that
    # deeper trees capture Store/Dept/holiday interactions much better.
    param_grid = {
        'depth': [6, 8, 10],
        'learning_rate': [0.03],
        'l2_leaf_reg': [3, 5],
    }
    tuning_iterations = 1800
    early_stopping_rounds = 50

    best_wmae = np.inf
    best_params = {}
    results = []

    # Reuse the cutoff-based origin-style train/validation matrices from CatBoost_CV.
    mlflow.log_params({
        'search_depth_values': str(param_grid['depth']),
        'search_learning_rate_values': str(param_grid['learning_rate']),
        'search_l2_leaf_reg_values': str(param_grid['l2_leaf_reg']),
        'tuning_iterations': tuning_iterations,
        'early_stopping_rounds': early_stopping_rounds,
        'tuning_val_weeks': ORIGIN_HORIZON_WEEKS,
        'tuning_features_count': len(CV_FEATURES),
        'tuning_training_feature_contract': 'origin_style',
        'tuning_cutoff': str(pd.Timestamp('2012-01-27').date()),
        'tuning_uses_holiday_sample_weights': True,
    })

    for trial_id, (depth, lr, l2) in enumerate(
        product(param_grid['depth'], param_grid['learning_rate'], param_grid['l2_leaf_reg']),
        start=1,
    ):
        params_dict = {'depth': depth, 'learning_rate': lr, 'l2_leaf_reg': l2}
        run_name = f'CatBoost_Tuning_trial_{trial_id:02d}_d{depth}_lr{lr}_l2{l2}'

        with mlflow.start_run(run_name=run_name, nested=True):
            mlflow.log_params(params_dict)
            mlflow.log_param('iterations', tuning_iterations)
            mlflow.log_param('loss_function', 'MAE')
            mlflow.log_param('early_stopping_rounds', early_stopping_rounds)
            mlflow.log_param('features_count', len(CV_FEATURES))
            mlflow.log_param('uses_holiday_sample_weights', True)

            m = CatBoostRegressor(
                iterations=tuning_iterations,
                depth=depth,
                learning_rate=lr,
                l2_leaf_reg=l2,
                loss_function='MAE',
                random_seed=SEED,
                verbose=False,
                early_stopping_rounds=early_stopping_rounds,
            )
            m.fit(
                X_tr_t, y_tr_t,
                cat_features=cat_idx_cv,
                sample_weight=w_tr_t,
                eval_set=cb.Pool(X_val_t, y_val_t, cat_features=cat_idx_cv, weight=w_val_t),
            )
            preds = m.predict(X_val_t)
            score = wmae(y_val_t.values, preds, hol_t.values)
            best_iteration_raw = m.get_best_iteration()
            best_iteration = (
                int(best_iteration_raw) + 1
                if best_iteration_raw is not None and best_iteration_raw >= 0
                else tuning_iterations
            )

            mlflow.log_metric('tuning_wmae', score)
            mlflow.log_metric('best_iteration', best_iteration)

        trial_result = {
            'trial_id': trial_id,
            **params_dict,
            'iterations': tuning_iterations,
            'best_iteration': best_iteration,
            'wmae': score,
        }
        results.append(trial_result)

        if score < best_wmae:
            best_wmae = score
            best_params = params_dict.copy()
            best_params['iterations'] = tuning_iterations
            best_params['best_iteration'] = int(best_iteration)
        print(f'trial={trial_id:02d} depth={depth} lr={lr} l2={l2}  ->  WMAE={score:,.2f}')

    tuning_df = pd.DataFrame(results).sort_values('wmae')
    tuning_df.to_csv('catboost_tuning_results.csv', index=False)

    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metric('best_tuning_wmae', best_wmae)
    mlflow.log_artifact('catboost_tuning_results.csv')

    fig, ax = plt.subplots(figsize=(10, 4))
    tuning_df.reset_index(drop=True)['wmae'].plot(ax=ax, marker='o')
    ax.set_title('Hyperparameter Search - WMAE per Config')
    ax.set_xlabel('Config Rank')
    ax.set_ylabel('WMAE')
    plt.tight_layout()
    plt.savefig('catboost_tuning.png', dpi=120, bbox_inches='tight')
    mlflow.log_artifact('catboost_tuning.png')
    plt.show()

    print()
    print(f'Best params: {best_params}  ->  WMAE={best_wmae:,.2f}')
    display(tuning_df)


## Final Model + Pipeline + Model Registry

The final CatBoost model uses the best raw configuration from the depth comparison: depth 10 with origin-style seasonal features. The pipeline accepts raw merged Walmart rows, applies origin-style feature engineering, preserves CatBoost-native missing values, and predicts `Weekly_Sales`.


In [ ]:
class WalmartOriginFeatureTransformerNoFill(BaseEstimator, TransformerMixin):
    """Transform raw merged Walmart rows into origin-style features without imputation."""
    def __init__(self, feature_cols=None):
        self.feature_cols = feature_cols

    def fit(self, X, y=None):
        df = X.copy()
        if TARGET not in df.columns:
            if y is None:
                raise ValueError('Training data or y must include Weekly_Sales for origin lag statistics.')
            df[TARGET] = np.asarray(y)

        self.feature_cols_ = list(self.feature_cols or ALL_FEATURES_ORIGIN)
        invalid_features = [c for c in self.feature_cols_ if c not in ALL_FEATURES_ORIGIN]
        if invalid_features:
            raise ValueError(f'Only ALL_FEATURES_ORIGIN columns are allowed. Invalid columns: {invalid_features}')

        self.history_ = df.copy()
        return self

    def transform(self, X):
        df = X.copy().reset_index(drop=True)
        df['_Input_Order'] = np.arange(len(df))

        df_base = build_all_features(df.drop(columns=[TARGET], errors='ignore'), add_lags=False)
        df_fe = add_origin_style_features(df_base, self.history_, origin_date=self.history_['Date'].max())
        df_fe = df_fe.sort_values('_Input_Order').reset_index(drop=True)

        for col in self.feature_cols_:
            if col not in df_fe.columns:
                df_fe[col] = np.nan

        return df_fe[self.feature_cols_].copy()


class FittedCatBoostWrapper(BaseEstimator):
    """Sklearn-compatible predict wrapper around a fitted CatBoostRegressor."""
    def __init__(self, model=None):
        self.model = model

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return self.model.predict(X)


In [ ]:
FINAL_MODEL_NAME = 'WalmartSales_CatBoost'
FINAL_FEATURES = [c for c in ORIGIN_FEATURES if c in train_model.columns]

assert set(FINAL_FEATURES).issubset(set(ALL_FEATURES_ORIGIN))
assert not (FORBIDDEN_FINAL_FEATURES & set(FINAL_FEATURES))

if 'best_params' not in globals() or not best_params:
    raise RuntimeError('Run CatBoost_Tuning before final training.')

FINAL_ITERATIONS = max(
    1,
    int(best_params.get('best_iteration', best_params['iterations'])),
)
FINAL_CAT_IDX = [
    FINAL_FEATURES.index(c)
    for c in CATBOOST_CAT_FEATURES
    if c in FINAL_FEATURES
]

print('Selected parameters:', best_params)
print('Final iterations:', FINAL_ITERATIONS)
print('Training shape:', train_model[FINAL_FEATURES].shape)

final_model = CatBoostRegressor(
    iterations=FINAL_ITERATIONS,
    depth=int(best_params['depth']),
    learning_rate=float(best_params['learning_rate']),
    l2_leaf_reg=float(best_params['l2_leaf_reg']),
    loss_function='MAE',
    random_seed=SEED,
    verbose=100,
)
final_model.fit(
    train_model[FINAL_FEATURES],
    train_model[TARGET],
    cat_features=FINAL_CAT_IDX,
    sample_weight=sample_weight,
)

feature_transformer = WalmartOriginFeatureTransformerNoFill(
    feature_cols=FINAL_FEATURES,
).fit(train_clean)

pipeline = Pipeline([
    ('features', feature_transformer),
    ('catboost', FittedCatBoostWrapper(final_model)),
])

X_test_probe = feature_transformer.transform(test_clean)
direct_probe = final_model.predict(X_test_probe)
pipeline_probe = pipeline.predict(test_clean)

assert list(X_test_probe.columns) == FINAL_FEATURES
assert len(pipeline_probe) == len(test_clean)
assert np.allclose(direct_probe, pipeline_probe)

with mlflow.start_run(run_name='CatBoost_Final') as run:
    mlflow.log_params({
        'model_family': 'CatBoost',
        'registered_model_name': FINAL_MODEL_NAME,
        'selection_source': 'CatBoost_Tuning',
        'depth': int(best_params['depth']),
        'iterations': FINAL_ITERATIONS,
        'learning_rate': float(best_params['learning_rate']),
        'l2_leaf_reg': float(best_params['l2_leaf_reg']),
        'loss_function': 'MAE',
        'feature_contract': 'raw_merged_to_weekly_sales',
        'training_feature_contract': 'origin_style',
        'uses_holiday_sample_weights': True,
        'preserves_input_order': True,
        'features_count': len(FINAL_FEATURES),
    })
    mlflow.log_metric('selection_wmae', float(best_wmae))
    mlflow.log_metric('final_holdout_wmae', float(best_wmae))
    mlflow.set_tags({
        'pipeline_contract': 'raw_merged_dataframe_to_weekly_sales',
        'winner_selected_by': 'minimum_validation_wmae',
    })
    mlflow.sklearn.log_model(
        pipeline,
        name='catboost_pipeline',
        registered_model_name=FINAL_MODEL_NAME,
        serialization_format='cloudpickle',
    )

print(f'Selected validation WMAE: {best_wmae:,.2f}')
print(f'Run ID: {run.info.run_id}')
print(f'Model registered as: {FINAL_MODEL_NAME}')


## Depth Comparison Summary

A focused depth experiment was run to test whether deeper CatBoost trees can recover Store/Dept seasonal structure without explicit origin lag features.

| Variant | Validation WMAE |
|---|---:|
| depth=5 + origin features | 1,722.82 |
| depth=5 + BASE_FEATURES only | 3,886.38 |
| depth=10 + origin features | 1,459.06 |
| depth=10 + BASE_FEATURES only | 2,552.17 |

Depth 10 substantially improved CatBoost, especially when paired with origin-style seasonal features. However, explicit `Sales_Lag52_Origin` remained essential: deeper trees partially learned Store/Dept/WeekOfYear interactions, but they did not fully replace the same-week-last-year signal. Based on this result, depth 10 was added to the normal tuning grid and used for the final pipeline.
